# Daily Weather Ingestion Job for 1-Hour Forecasting

This notebook ingests the **daily hourly weather forecast** used by the short-term (1-hour ahead) prediction pipeline.

It retrieves a minimal set of weather variables from Open-Meteo for the Toronto area, specifically:

- hourly temperature
- hourly apparent temperature

The goal is to maintain a lightweight and reliable weather input source for the hourly forecasting branch, while also preserving the raw JSON payload for traceability.

This notebook represents the **daily weather ingestion layer for the 1-hour prediction workflow**.

## Process Overview

This notebook performs the following steps:

### 1. Query the Open-Meteo API
The job requests the hourly weather forecast for the current day using:
- latitude/longitude for downtown Toronto
- local timezone (`America/Toronto`)
- hourly temperature variables only

### 2. Save the raw JSON payload
The downloaded response is stored in DBFS to preserve:
- ingestion traceability
- reproducibility
- debugging capability

### 3. Build structured hourly weather records
The JSON response is transformed into a tabular format with:

- year
- month
- day
- hour
- local weather timestamp
- temperature
- apparent temperature

### 4. Write to a Unity Catalog bronze table
The structured records are appended into:
- `workspace.default.bronze_weather_hourly_minimal`

### 5. Refresh the latest weather view
A latest-per-hour view is created so downstream jobs can always retrieve the newest weather record for each date-hour combination:

- `workspace.default.vw_weather_hourly_minimal_latest`

In [0]:
# Databricks notebook source
# ============================================================
# WEATHER DAILY INGESTION JOB - MINIMAL VERSION
# Serverless + Unity Catalog safe
# Only downloads:
#   year, month, day, hour, temperature_2m, apparent_temperature
# ============================================================

from pyspark.sql import functions as F
import requests
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

# ------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------
UC_CATALOG = "workspace"
UC_SCHEMA  = "default"

TBL_WEATHER = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_weather_hourly_minimal"

TIMEZONE = "America/Toronto"
LATITUDE = 43.65
LONGITUDE = -79.38

# Optional raw json path
RAW_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather"

# ------------------------------------------------------------
# 1) HELPERS
# ------------------------------------------------------------
def fetch_weather():
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "hourly": "temperature_2m,apparent_temperature",
        "timezone": TIMEZONE,
        "forecast_days": 1
    }

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return response.json()

def write_raw_json(payload: dict):
    now_local = datetime.now(ZoneInfo(TIMEZONE))
    yyyy = now_local.strftime("%Y")
    mm   = now_local.strftime("%m")
    dd   = now_local.strftime("%d")
    ts   = now_local.strftime("%Y%m%dT%H%M%S")

    out_dir = f"{RAW_DIR}/forecast_today/year={yyyy}/month={mm}/day={dd}"
    out_file = f"{out_dir}/forecast_today_{ts}.json"

    dbutils.fs.put(out_file, json.dumps(payload, ensure_ascii=False), overwrite=True)
    print(f"RAW saved: {out_file}")

def build_records(data: dict):
    hourly = data.get("hourly", {})
    times = hourly.get("time", [])
    temps = hourly.get("temperature_2m", [])
    app_temps = hourly.get("apparent_temperature", [])

    records = []
    for i in range(len(times)):
        dt = datetime.fromisoformat(times[i])
        records.append({
            "year": dt.year,
            "month": dt.month,
            "day": dt.day,
            "hour": dt.hour,
            "weather_ts_local": times[i],
            "temperature_2m_c": temps[i] if i < len(temps) else None,
            "apparent_temperature_c": app_temps[i] if i < len(app_temps) else None
        })
    return records

# ------------------------------------------------------------
# 2) INGEST
# ------------------------------------------------------------
import json

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")

print("========================================")
print("Weather minimal ingestion started")
print(f"Target table: {TBL_WEATHER}")
print("========================================")

data = fetch_weather()
write_raw_json(data)

records = build_records(data)

pdf = pd.DataFrame(records)
pdf = pdf.where(pd.notnull(pdf), None)

df = spark.createDataFrame(pdf)

df = (
    df.withColumn("weather_ts_local", F.to_timestamp("weather_ts_local"))
      .withColumn(
          "ingested_at_utc",
          F.current_timestamp()
      )
)

(
    df.write
      .format("delta")
      .mode("append")
      .saveAsTable(TBL_WEATHER)
)

print(f"Weather saved: {TBL_WEATHER}")

# ------------------------------------------------------------
# 3) OPTIONAL VIEW: LATEST PER HOUR
# ------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {UC_CATALOG}.{UC_SCHEMA}.vw_weather_hourly_minimal_latest AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY year, month, day, hour
               ORDER BY ingested_at_utc DESC
           ) AS rn
    FROM {TBL_WEATHER}
)
SELECT *
FROM ranked
WHERE rn = 1
""")

print("View refreshed")

try:
    display(spark.table(TBL_WEATHER).limit(10))
except Exception as e:
    print(f"Could not display sample: {e}")

print("========================================")
print("Weather minimal ingestion finished")
print("========================================")

## Outputs

This notebook produces the following outputs:

### 1. Raw JSON weather files
The original Open-Meteo response is stored in DBFS under the raw weather ingestion path for auditability and future reuse.

---

### 2. Bronze hourly weather table
Structured hourly weather records are appended to:

- `workspace.default.bronze_weather_hourly_minimal`

This table contains:
- local hourly timestamp
- temperature
- apparent temperature
- ingestion timestamp

---

### 3. Latest-per-hour weather view
The notebook refreshes:

- `workspace.default.vw_weather_hourly_minimal_latest`

This view exposes the most recently ingested weather record for each `(year, month, day, hour)` combination and is used by downstream short-term prediction jobs.

## Key Insights and Summary

### 1. This notebook provides the weather context for hourly forecasting
The 1-hour prediction pipeline requires weather conditions aligned with the target forecast hour. This notebook supplies that information in a lightweight and operationally efficient format.

---

### 2. The design is intentionally minimal
Only the most relevant weather variables for the hourly model are ingested:
- temperature
- apparent temperature

This keeps the pipeline simple, fast, and easy to maintain.

---

### 3. Historical traceability is preserved
Even though the notebook is minimal, it still stores:
- raw JSON files
- timestamped bronze records
- latest-per-hour view logic

This supports reproducibility and operational debugging.

---

### 4. The latest view simplifies downstream joins
Instead of requiring downstream jobs to deduplicate weather snapshots, the notebook provides a ready-to-use latest view keyed by hour.

This reduces complexity in the short-term prediction pipeline.

---

### 5. Business relevance
Weather is an important external driver of bike-sharing demand. By maintaining a fresh daily weather source, this notebook helps improve the quality of short-term station-level predictions.

In practical terms, this notebook is the **daily weather input provider for the 1-hour forecasting branch**.